# Pilot-B evaluation — meta-llama/Llama-3.2-3B-Instruct
**Real evaluation run** on the frozen Pilot-B set: 70 human-reviewed families × E/R/C = 210 prompts. Pre-registered plan: `pilot_b/PREREGISTRATION.md` (decision D035).
Frozen protocol: prompt v2, greedy non-thinking decoding, NF4 + FP16, scorer v1.0.0. **Do not change any setting.**
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Takes about 25–35 minutes. A results ZIP downloads at the end; send it unchanged.
Optional: set `USE_DRIVE = True` in the next cell so a disconnect can resume (re-run all with the same settings).


In [ ]:
import os, sys, json, base64, io, zipfile, hashlib, subprocess, traceback
from pathlib import Path
MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
RUN_NAME = 'pilotb-Llama-3.2-3B-Instruct'
USE_DRIVE = False
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
PACKAGE = Path('/content/hinglish_pilotb') if Path('/content').exists() else Path.cwd() / 'hinglish_pilotb'
PACKAGE.mkdir(parents=True, exist_ok=True)
blob = base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest() == 'c8f843f7a33e36ece96791adde507d5a4e67f9493d9e6b213a33f9d886d7f7d8'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
sys.path.insert(0, str(PACKAGE))
from hinglish_pilot.engine import default_config, environment, read, write, run, utc
from hinglish_pilot.data import validate
from hinglish_pilot.report import analyze, export
from secondary_analysis.report_secondary import analyze as analyze_secondary
DATASET, SPLITS = read(PACKAGE / 'pilot_b/dataset.json'), read(PACKAGE / 'pilot_b/splits.json')
print(json.dumps(validate(DATASET, SPLITS, 'pilot_b'), indent=1))
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS = Path('/content/drive/MyDrive/Hinglish_PilotB')
else:
    RUNS = PACKAGE / 'runs'
RUN_DIR = RUNS / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
if True:
    # Gated model: needs a Hugging Face read token (Colab secret HF_TOKEN, or paste when asked).
    token = ''
    try:
        from google.colab import userdata
        token = (userdata.get('HF_TOKEN') or '').strip()
        print('HF_TOKEN read from Colab secret')
    except Exception as error:
        print('HF_TOKEN Colab secret not available (' + type(error).__name__ + '): turn on Notebook access for it in the key sidebar, or paste the token now.')
    import getpass
    while not token.startswith('hf_'):
        token = getpass.getpass('Paste your Hugging Face read token (starts with hf_): ').strip()
    os.environ['HF_TOKEN'] = token
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface-hub==0.30.2'], check=True)
    from huggingface_hub import login, auth_check
    login(token=token, add_to_git_credential=False)
    auth_check(MODEL_ID, token=token)  # clear error if this token lacks access
    print('Access to', MODEL_ID, 'confirmed')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements-colab.txt')], check=True)
write(RUNS / f'{RUN_NAME}_environment.json', environment())


In [ ]:
from hinglish_pilot.hf_backend import resolve_config, HFBackend
CONFIG = resolve_config(default_config('pilot_b', MODEL_ID), RUN_DIR)
backend = HFBackend(CONFIG)
write(RUNS / f'{RUN_NAME}_model_load_report.json', backend.load_report)
print('Model loaded on', backend.metadata['hardware'], 'revision', CONFIG['model_revision'])


In [ ]:
progress = lambda done, total, item: print(f'{done}/{total}: {item["rendering_id"]} {item["latency_seconds"]:.1f}s', flush=True)
manifest = run(RUN_DIR, DATASET, SPLITS, CONFIG, backend, progress=progress)
print('Run status:', manifest['status'], manifest['completed'], '/', manifest['expected'])
metrics, _ = analyze(RUN_DIR)
print((RUN_DIR / 'summary.md').read_text())
if manifest['status'] == 'COMPLETE':
    _, folder = analyze_secondary(RUN_DIR)
    print((folder / 'secondary_summary.md').read_text())


## Download
If the run stopped early, re-run all cells with the same settings: completed prompts are kept and only missing ones run. Then run this cell.


In [ ]:
results = export(RUN_DIR)
archive = PACKAGE / 'PilotB_results_Llama-3.2-3B-Instruct.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(results, results.name)
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file() and 'secondary_diagnostic_' in str(path):
            z.write(path, str(path.relative_to(RUN_DIR)))
    for path in sorted(RUNS.glob(f'{RUN_NAME}_*.json')):
        z.write(path, path.name)
print('Return:', archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Local: ZIP saved on disk.')
